# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charitha-05/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip -q install duckdb

In [7]:
from huggingface_hub import get_token

HF_TOKEN = get_token()

print("Hugging Face token available:", HF_TOKEN is not None)

Hugging Face token available: True


In [8]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB is connected to Hugging Face.")

DuckDB is connected to Hugging Face.


In [15]:
DAILY_MAR = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

test = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {DAILY_MAR}
""").df()

display(test)

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row in the modeling dataset will represent one content page for one client at one monthly decision point.

For development, I will use March 2026 as the decision month. The modeling features will be constructed only from information that would have been available on or before the decision point.

Historical search-performance signals will be aggregated over predefined lookback windows, while content attributes will be evaluated as of the decision point.

Future performance will be kept separate from the feature window so that it can later be used to construct the refresh-priority outcome without introducing temporal leakage.

The underlying daily warehouse data is at the grain of one client × one content page × one report date. This daily data will be aggregated to the modeling grain described above.

In [12]:
q1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {DAILY_MAR}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

grain_check = con.sql(q1).df()

display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [13]:
q2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {DAILY_MAR}
"""

date_check = con.sql(q2).df()

display(date_check)

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Features

The initial feature set will contain:

1. `impressions_90d` — historical search impressions for the content page.
2. `clicks_90d` — historical search clicks for the content page.
3. `avg_position_90d` — historical average search position.
4. `content_age_days` — age of the content at the decision point.
5. `impression_trend` — change in impressions between two historical 30-day periods before the decision point.

These features represent historical search performance and content characteristics that would be available when the refresh-priority decision is made.

### Label

The label will be a refresh-priority proxy based on measurable future search-performance outcomes.

The warehouse does not provide a direct human-labelled field stating that a page needs a refresh. Therefore, the eventual target will be explicitly constructed from future outcomes and kept separate from the feature window.

### Context

The following fields will be retained as context for identifying, filtering, grouping, and interpreting observations:

- `client_hash_id`
- `content_hash_id`
- `content_type`
- `main_intent`
- `competition_level`
- relevant data-availability indicators

Client and content identifiers are required for grouping and validation but will not be used as predictive features.

### Excluded

The following will be excluded from the predictive feature set:

- `client_hash_id` and `content_hash_id` as model inputs, because they are identifiers rather than meaningful predictive attributes.
- Future performance variables, because they would not be available at the decision point.
- Any variable directly derived from the future refresh-priority outcome, because it would create target leakage.
- `trend_direction` and `trend_pct`, because these are outcome/trend-derived fields and should not be used as independent predictive inputs.
- Any post-decision information, because it would violate the temporal availability requirement.

The feature set will contain only information available at the decision point.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Verification 1 — Grain**

In [16]:
q1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {DAILY_MAR}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

grain_check = con.sql(q1).df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


**Verification 2 — Row count and date window**

In [17]:
q2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {DAILY_MAR}
"""

date_check = con.sql(q2).df()

display(date_check)

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


**Verification 3 — GSC availability**

In [18]:
q3 = f"""
SELECT
    gsc_data_available,
    COUNT(*) AS row_count
FROM {DAILY_MAR}
GROUP BY gsc_data_available
ORDER BY gsc_data_available
"""

availability_check = con.sql(q3).df()

display(availability_check)

,gsc_data_available,row_count
0,False,6230317
1,True,3611061


**Verification 4 — Explicitly check usable GSC rows**

In [19]:
gsc_available = con.sql(f"""
SELECT
    COUNT(*) AS gsc_available_rows
FROM {DAILY_MAR}
WHERE gsc_data_available IS TRUE
""").df()

display(gsc_available)

,gsc_available_rows
0,3611061


### Verification findings

The March 2026 partition contains 9,841,378 daily content-performance records covering March 1 through March 31, 2026.

The grain check returned no duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id`, supporting the claimed daily warehouse grain.

GSC availability is not uniform across the March observations. There are 3,611,061 rows where GSC data is available and 6,230,317 rows where it is unavailable.

Therefore, GSC-derived features must be constructed with explicit availability handling rather than assuming that search data exists for every observation.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


The warehouse has several important limitations.

First, historical coverage is not balanced across clients. Different clients may have different amounts of usable historical data, so the amount of information available for a content page can vary across observations.

Second, GSC availability is not uniform across the data. The March 2026 verification showed that only a subset of daily observations has GSC data available. Therefore, search-performance features cannot be assumed to be available for every row.

Third, historical lookback windows can overlap across decision points. As a result, adjacent modeling observations may share historical information and should not be treated as fully independent.

Fourth, the warehouse does not contain a direct human-labelled outcome indicating that a page "needs a refresh." The target therefore has to be defined as a measurable proxy based on future performance. The resulting model should be interpreted as decision support for prioritization rather than as a direct measurement of an editorial refresh decision.

Finally, the model cannot establish that refreshing a page will cause performance to improve. It can identify pages whose observed characteristics and historical performance make them higher-priority candidates for review, but causal impact would require a separate experimental design.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.